# 05b Column Role Dictionary Patch

Patch semantic role/family/timing errors from step 05. This notebook does not perform modeling, feature engineering, prediction, SHAP, Optuna, row exclusion, duplicate removal, or model-ready dataset creation.

In [1]:
from pathlib import Path
from datetime import datetime
from zipfile import ZipFile, ZIP_DEFLATED
import subprocess, json, re
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 240)

STEP = "05b_column_role_dictionary_patch_260513"
PREV_STEP = "05_column_role_leakage_timing_audit_260513"
EXPECTED_ROOT_TEXT = r"C:\Code\ott-churn-prediction"
NOW = datetime.now()
RUN_TS = NOW.strftime("%Y%m%d_%H%M%S")

def find_park():
    cwd = Path.cwd().resolve()
    source_name = "(광일)Membership_v2_with_derived_features.csv"
    for cand in [cwd, *cwd.parents]:
        if (cand / "data" / source_name).exists() and cand.name == "park.ingyeom":
            return cand.resolve()
        if (cand / "park.ingyeom" / "data" / source_name).exists():
            return (cand / "park.ingyeom").resolve()
    return (cwd / "park.ingyeom").resolve()

PARK = find_park()
ROOT = PARK.parent.resolve()
SRC = PARK / "data" / "(광일)Membership_v2_with_derived_features.csv"
PREV_DIR = PARK / "reports" / "audits" / PREV_STEP
OUT_BASE = PARK / "reports" / "audits" / STEP
NB_PATH = PARK / "notebook" / STEP / f"{STEP}.ipynb"
NOTE = PARK / "note.md"
ZIP_DIR = PARK / "zip"
ZIP_PATH = ZIP_DIR / f"{STEP}_review_package.zip"
REQ_PREV = [
    "05_full_column_inventory.csv", "05_column_role_dictionary.csv", "05_timing_audit.csv",
    "05_leakage_suspect_audit.csv", "05_human_review_required_columns.csv",
    "05_baseline_ladder_feature_family_policy.csv", "05_recommended_feature_set_contracts.csv",
    "05_forbidden_drop_columns.csv", "05_review_required_columns.csv",
    "05_conservative_safe_candidate_columns.csv", "05_redundancy_and_naming_risk_audit.csv",
    "05_final_checks.csv", "README.md",
]
REQ_OUT = [
    "05b_input_validation.csv", "05b_detected_issues_from_05.csv",
    "05b_canonical_column_role_dictionary.csv", "05b_column_role_patch_log.csv",
    "05b_canonical_timing_audit.csv", "05b_canonical_leakage_suspect_audit.csv",
    "05b_true_human_review_required_columns.csv", "05b_human_review_summary.csv",
    "05b_canonical_recommended_feature_set_contracts.csv",
    "05b_conservative_safe_candidate_columns.csv", "05b_review_required_columns.csv",
    "05b_forbidden_drop_columns.csv", "05b_role_and_status_summary.csv",
    "05b_downstream_handoff_policy.csv", "05b_safe_unsafe_wording.csv",
    "05b_open_risks_for_next_steps.csv",
]

def inside(child, parent):
    try:
        Path(child).resolve().relative_to(Path(parent).resolve())
        return True
    except ValueError:
        return False

def read_csv(path):
    last = None
    for enc in ["utf-8-sig", "utf-8", "cp949", "euc-kr"]:
        try:
            return pd.read_csv(path, encoding=enc)
        except Exception as e:
            last = e
    raise last

for p in [NB_PATH.parent, OUT_BASE, ZIP_DIR]:
    p.mkdir(parents=True, exist_ok=True)
payload = [p for p in OUT_BASE.iterdir() if p.name != ".ipynb_checkpoints"] if OUT_BASE.exists() else []
if payload:
    OUT = OUT_BASE / f"run_{RUN_TS}"
    OUT.mkdir(parents=True, exist_ok=False)
    out_mode = "run_subfolder_created_because_base_output_folder_was_not_empty"
else:
    OUT = OUT_BASE
    OUT.mkdir(parents=True, exist_ok=True)
    out_mode = "base_output_folder_used"

git_result = subprocess.run(["git", "rev-parse", "--show-toplevel"], cwd=str(ROOT), text=True, capture_output=True)
actual_repo_root = git_result.stdout.strip().replace("\\", "/") if git_result.returncode == 0 else ""
expected_norm = EXPECTED_ROOT_TEXT.replace("\\", "/").rstrip("/").lower()
actual_norm = actual_repo_root.rstrip("/").lower()
repo_root_warning = actual_norm != expected_norm
warnings_log = []
if repo_root_warning:
    warnings_log.append({"warning_type": "actual_repo_root_differs_from_requested", "detail": f"requested={EXPECTED_ROOT_TEXT}; actual={actual_repo_root}"})

source_mtime_before = SRC.stat().st_mtime if SRC.exists() else None
prev_mtimes_before = {name: (PREV_DIR / name).stat().st_mtime for name in REQ_PREV if (PREV_DIR / name).exists()}

df = read_csv(SRC)
prev = {name: read_csv(PREV_DIR / name) for name in REQ_PREV if name.endswith(".csv") and (PREV_DIR / name).exists()}
prev_readme_text = (PREV_DIR / "README.md").read_text(encoding="utf-8", errors="replace") if (PREV_DIR / "README.md").exists() else ""
inventory = prev["05_full_column_inventory.csv"]
old_roles = prev["05_column_role_dictionary.csv"]
old_timing = prev["05_timing_audit.csv"]
old_contracts = prev["05_recommended_feature_set_contracts.csv"]
old_human = prev["05_human_review_required_columns.csv"]

dupe_user_extra = int(df.duplicated("USER_KEY").sum()) if "USER_KEY" in df else np.nan
dupe_full = int(df.duplicated().sum())
cross_promo = int((df.groupby("USER_KEY")["is_promotion"].nunique(dropna=True) > 1).sum()) if {"USER_KEY", "is_promotion"}.issubset(df.columns) else np.nan

input_validation_rows = []
def add_val(check, status, detail):
    input_validation_rows.append({"check_name": check, "status": status, "detail": detail})
add_val("source_csv_exists", "PASS" if SRC.exists() else "FAIL", str(SRC))
add_val("previous_05_folder_exists", "PASS" if PREV_DIR.exists() else "FAIL", str(PREV_DIR))
missing_prev = [name for name in REQ_PREV if not (PREV_DIR / name).exists()]
add_val("all_required_05_input_files_exist", "PASS" if not missing_prev else "FAIL", "; ".join(missing_prev))
add_val("05_full_column_inventory_has_91_rows", "PASS" if len(inventory) == 91 else "FAIL", f"rows={len(inventory)}")
add_val("05_column_role_dictionary_has_91_rows", "PASS" if len(old_roles) == 91 else "FAIL", f"rows={len(old_roles)}")
add_val("05_timing_audit_has_91_rows", "PASS" if len(old_timing) == 91 else "FAIL", f"rows={len(old_timing)}")
add_val("source_csv_has_23343_rows_91_columns", "PASS" if df.shape == (23343, 91) else "FAIL", f"shape={df.shape}")
add_val("USER_KEY_duplication_recomputed", "PASS", f"duplicated_USER_KEY_extra_rows={dupe_user_extra}; duplicated_full_row_count={dupe_full}; cross_promotion_USER_KEY_overlap={cross_promo}")
add_val("actual_repo_root_recorded", "WARNING" if repo_root_warning else "PASS", actual_repo_root)
add_val("all_planned_outputs_inside_park_ingyeom", "PASS" if all(inside(OUT / n, PARK) for n in REQ_OUT + ["05b_final_checks.csv", "README.md"]) else "FAIL", str(OUT))
pd.DataFrame(input_validation_rows).to_csv(OUT / "05b_input_validation.csv", index=False, encoding="utf-8-sig")

old_idx = old_roles.set_index("column_name")
actual_genre_cols = {
    "genre_diversity_count", "action_adventure_ratio", "family_animation_ratio", "drama_ratio",
    "thriller_crime_ratio", "sf_fantasy_ratio", "comedy_ratio", "romance_ratio", "horror_ratio",
    "documentary_ratio", "historical_war_ratio", "other_ratio"
}
new_movie_cols = {"new_movie_in_90d_ratio", "new_movie_in_180d_ratio", "new_movie_in_365d_ratio"}
usage_ratio_cols = {"active_ratio", "avg_rewatch_ratio", "weekend_watch_ratio", "watch_ratio_under_1m", "watch_ratio_under_5m"}
retention_ratio_cols = {"retention_w2_ratio", "retention_w3_ratio"}
diff_cols = {"diff_between_w2_w1", "diff_between_w3_w1", "diff_between_w3_w2"}
cold_cols = {"is_cold_start_3d", "is_cold_start_7d"}
usage_summary_cols = {
    "unique_movie", "watch_days", "watch_per_day", "avg_watch_time(min)", "median_watch_time(min)",
    "std_watch_time(min)", "avg_daily_watch_time(min)", "max_watch_time(min)", "max_daily_watch_time(min)",
    "max_daily_sessions", "avg_gap_between_watch_days", "max_inactive_gap_days", "movie_per_active_day",
    "max_day_share", "day_count_over_3times"
}
membership_context_cols = {
    "product_code", "price", "billing_method", "max_screen", "payment_device", "is_user_verified",
    "gender", "age", "is_standard", "is_premium", "age_group", "is_female", "is_male",
    "payment_is_mobile", "payment_is_pc", "payment_is_android", "payment_is_ios"
}
registration_context_cols = {"reg_hour", "reg_is_weekend", "reg_hour_morning", "reg_hour_afternoon", "reg_hour_evening", "reg_hour_night"}
week_allowed_cols = {
    "avg_gap_w1_watch_days", "avg_gap_w2_watch_days", "avg_gap_w3_watch_days",
    "watch_time(min)_w1", "watch_time(min)_w2", "watch_time(min)_w3",
    "watch_session_w1", "watch_session_w2", "watch_session_w3",
    "is_w1_over_50pct", "is_w2_over_50pct", "is_w3_over_50pct",
    "is_only_w1", "is_only_w2", "is_only_w3"
}

def old_value(col, field):
    return old_idx.loc[col, field] if col in old_idx.index and field in old_idx.columns else ""

def patched_for(col):
    old = old_idx.loc[col].to_dict()
    role = old["primary_role"]; fam = old["feature_family"]; sub = old["sub_family"]; timing = old["timing_family"]
    overall = old["allowed_for_overall_model_candidate"]; group = old["allowed_for_groupwise_model_candidate"]
    lm = old["allowed_for_baseline_ladder_membership_only"]; la = old["allowed_for_baseline_ladder_activation"]
    lw2 = old["allowed_for_baseline_ladder_retention_w2"]; lw3 = old["allowed_for_baseline_ladder_retention_w3"]
    lc = old["allowed_for_baseline_ladder_content"]; seg = old["allowed_for_segment_design_candidate"]
    conf = old.get("required_user_confirmation", "yes"); fut = old.get("future_step_to_resolve", "06_common_preprocessing_and_final_cohort_260513")
    reason = old["reason"]
    def set_all(o=None, g=None, ml=None, ac=None, w2=None, w3=None, co=None, sg=None):
        nonlocal overall, group, lm, la, lw2, lw3, lc, seg
        if o is not None: overall = o
        if g is not None: group = g
        if ml is not None: lm = ml
        if ac is not None: la = ac
        if w2 is not None: lw2 = w2
        if w3 is not None: lw3 = w3
        if co is not None: lc = co
        if sg is not None: seg = sg
    if col == "USER_KEY":
        role, fam, sub, timing, conf, fut = "id", "identifier", "row_or_user_key", "id_target_split", "no", "never_use_as_model_feature"
        set_all("no", "no", "no", "no", "no", "no", "no", "no")
        reason = "Identifier; row-level/subscription-event-level analysis is not unique-user-level. Never use as model feature."
    elif col == "is_repurchase":
        role, fam, sub, timing, conf, fut = "target", "target", "repurchase_positive_class", "id_target_split", "no", "target_contract_established"
        set_all("no", "no", "no", "no", "no", "no", "no", "no")
        reason = "Target; positive class means repurchase. Never use as model feature."
    elif col == "is_promotion":
        role, fam, sub, timing, conf, fut = "split", "split_variable", "promotion_row_split", "id_target_split", "no", "overall_model_comparison_only_if_used"
        set_all("yes", "no", "no", "no", "no", "no", "no", "review")
        reason = "Top-level split variable. It may be used only for overall comparison, and must be excluded inside promotion-only/non-promotion-only models."
    elif col == "reg_date":
        role, fam, sub, timing, conf, fut = "date_or_time_anchor", "subscription_anchor", "registration_date", "subscription_anchor_date", "yes", "confirm_allowed_date_derivations_only"
        set_all("no", "no", "no", "no", "no", "no", "no", "review")
        reason = "Registration date is an anchor, not a regular model feature."
    elif col == "end_date":
        role, fam, sub, timing, conf, fut = "timing_review_required", "subscription_end_timing_review", "end_date_ambiguous", "subscription_end_timing_review", "yes", "06_common_preprocessing_and_final_cohort_260513"
        set_all("review", "review", "no", "no", "no", "no", "no", "review")
        reason = "End date is unresolved: scheduled-at-scoring vs post-hoc actual end must be confirmed before modeling."
    elif col == "is_churn_prevented":
        role, fam, sub, timing, conf, fut = "leakage_suspect", "outcome_or_intervention_proxy", "churn_prevention_ambiguous", "post_outcome_or_target_review", "yes", "resolve_intervention_timing_before_modeling"
        set_all("review", "review", "no", "no", "no", "no", "no", "review")
        reason = "Could be past information or current-cycle intervention/outcome proxy; keep review until source timing is proven."
    elif col in retention_ratio_cols:
        role, fam, sub = "retention", "retention_ratio", col.replace("retention_", "")
        timing = "week1_to_week2_change" if col == "retention_w2_ratio" else "week1_to_week3_change"
        conf, fut = "no", "baseline_ladder_retention"
        set_all("yes", "yes", "no", "no", "yes" if col.endswith("w2_ratio") else "no", "yes" if col.endswith("w3_ratio") else "no", "no", "yes")
        reason = f"{col} is retention behavior, not genre. Name explicitly references allowed week window; allowed status corrected to yes under the day0-20 observation contract."
    elif col in diff_cols:
        role, fam, sub = "retention", "retention_change", col
        timing = {"diff_between_w2_w1": "week1_to_week2_change", "diff_between_w3_w1": "week1_to_week3_change", "diff_between_w3_w2": "week2_to_week3_change"}[col]
        conf, fut = "no", "baseline_ladder_retention_change"
        set_all("yes", "yes", "no", "no", "yes" if col == "diff_between_w2_w1" else "no", "yes" if col in {"diff_between_w3_w1", "diff_between_w3_w2"} else "no", "no", "yes")
        reason = "Week-to-week difference feature within week1 to week3 observation window; role/family corrected to retention_change."
    elif col in cold_cols:
        role, fam, sub = "activation", "activation_onboarding", col
        timing = "early_activation" if col == "is_cold_start_3d" else "week1_observation"
        conf, fut = "no", "baseline_ladder_activation"
        set_all("yes", "yes", "no", "yes", "no", "no", "no", "yes")
        reason = "Cold-start flag is early activation/onboarding behavior, not content recency; 3d/7d is inside the allowed observation window."
    elif col in usage_ratio_cols:
        role = "retention" if col == "active_ratio" else "aggregate_usage_review"
        fam = {"active_ratio": "usage_behavior_ratio", "avg_rewatch_ratio": "engagement_depth", "weekend_watch_ratio": "viewing_pattern", "watch_ratio_under_1m": "short_watch_behavior", "watch_ratio_under_5m": "short_watch_behavior"}[col]
        sub, timing, conf, fut = col, "all_period_ambiguous", "yes", "confirm_usage_ratio_window_before_modeling"
        set_all("review", "review", "no", "no", "no", "no", "no", "review")
        reason = f"{col} is a usage/behavior ratio, not a genre ratio. Construction window is not explicit, so it remains review."
    elif col in new_movie_cols:
        role, fam, sub, timing, conf, fut = "content_recency", "content_recency", col, "content_recency_window_ambiguous", "yes", "confirm_content_window_before_modeling"
        set_all("review", "review", "no", "no", "no", "no", "review", "review")
        reason = "New-movie ratio is content recency, not genre. Observation window must be confirmed before modeling."
    elif col == "avg_ott_release_year":
        role, fam, sub, timing, conf, fut = "content_recency", "content_metadata_recency", "avg_release_year", "content_recency_window_ambiguous", "yes", "confirm_content_window_before_modeling"
        set_all("review", "review", "no", "no", "no", "no", "review", "review")
        reason = "Average release year is content metadata/recency; construction window remains unresolved."
    elif col in actual_genre_cols:
        role, fam, sub, timing, conf, fut = "genre_ratio", "content_genre", col, "content_window_ambiguous", "yes", "confirm_content_window_before_modeling"
        set_all("review", "review", "no", "no", "no", "no", "review", "review")
        reason = "Actual genre/content preference column. Keep review until the observation window is documented."
    elif col in {"total_watch_count", "total_watch_time(min)"}:
        role, fam, sub, timing, conf, fut = "aggregate_usage_review", "usage_aggregate", "total_period_ambiguous", "all_period_ambiguous", "yes", "confirm_total_usage_window_before_modeling"
        set_all("review", "review", "no", "no", "no", "no", "no", "review")
        reason = "Total usage column remains review until proven to be day0-20 only."
    elif col == "recency":
        role, fam, sub, timing, conf, fut = "timing_review_required", "content_recency", "reference_date_review", "reference_date_review", "yes", "confirm_recency_reference_date"
        set_all("review", "review", "no", "no", "no", "no", "review", "review")
        reason = "Recency reference date is unresolved; it must be day21 scoring point or earlier."
    elif col in usage_summary_cols:
        role, fam, sub, timing, conf, fut = "aggregate_usage_review", "usage_behavior_summary", col, "all_period_ambiguous", "yes", "confirm_usage_window_before_modeling"
        set_all("review", "review", "no", "no", "no", "no", "no", "review")
        reason = "Usage summary lacks explicit week/window in the name; keep review until construction window is documented."
    elif col in membership_context_cols:
        role, fam, sub, timing, conf, fut = "membership_context", "membership_context", col, "static_or_pre_subscription_review", "yes", "confirm_metadata_semantics_before_modeling"
        set_all("review", "review", "review", "no", "no", "no", "no", "review")
        reason = "Membership/payment/demographic context needs semantic and scoring-time review before modeling."
    elif col in registration_context_cols:
        role, fam, sub, timing, conf, fut = "acquisition", "registration_context", col, "subscription_anchor_derived_review", "yes", "confirm_registration_context_semantics_before_modeling"
        set_all("review", "review", "review", "no", "no", "no", "no", "review")
        reason = "Registration-time derived context; review whether it is appropriate as a feature rather than anchor leakage."
    elif col in week_allowed_cols:
        if "_w1" in col or "w1_" in col or col.endswith("w1"):
            role, fam, sub, timing, conf, fut = "activation", "usage_observation", col, "week1_observation", "no", "baseline_ladder_activation"
            set_all("yes", "yes", "no", "yes", "no", "no", "no", "yes")
        elif "_w2" in col or "w2_" in col or col.endswith("w2"):
            role, fam, sub, timing, conf, fut = "retention", "usage_observation", col, "week2_observation", "no", "baseline_ladder_retention_w2"
            set_all("yes", "yes", "no", "no", "yes", "no", "no", "yes")
        else:
            role, fam, sub, timing, conf, fut = "retention", "usage_observation", col, "week3_observation", "no", "baseline_ladder_retention_w3"
            set_all("yes", "yes", "no", "no", "no", "yes", "no", "yes")
        reason = "Explicit week1-week3 observation feature under the established day0-20 window."
    patched = {
        "patched_primary_role": role, "patched_feature_family": fam, "patched_sub_family": sub, "patched_timing_family": timing,
        "patched_allowed_for_overall_model_candidate": overall, "patched_allowed_for_groupwise_model_candidate": group,
        "patched_allowed_for_baseline_ladder_membership_only": lm, "patched_allowed_for_baseline_ladder_activation": la,
        "patched_allowed_for_baseline_ladder_retention_w2": lw2, "patched_allowed_for_baseline_ladder_retention_w3": lw3,
        "patched_allowed_for_baseline_ladder_content": lc, "patched_allowed_for_segment_design_candidate": seg,
        "patched_reason": reason, "patched_required_user_confirmation": conf, "patched_future_step_to_resolve": fut,
    }
    changed_bits = []
    for old_field, new_field in [
        ("primary_role", "patched_primary_role"), ("feature_family", "patched_feature_family"), ("sub_family", "patched_sub_family"),
        ("timing_family", "patched_timing_family"), ("allowed_for_overall_model_candidate", "patched_allowed_for_overall_model_candidate"),
        ("allowed_for_groupwise_model_candidate", "patched_allowed_for_groupwise_model_candidate"),
    ]:
        if str(old.get(old_field, "")) != str(patched[new_field]):
            changed_bits.append(f"{old_field}: {old.get(old_field, '')} -> {patched[new_field]}")
    patched["patch_changed"] = "yes" if changed_bits else "no"
    patched["patch_change_summary"] = "; ".join(changed_bits)
    return patched

patched_rows = []
for _, row in old_roles.iterrows():
    patched_rows.append({**row.to_dict(), **patched_for(row["column_name"])})
canon = pd.DataFrame(patched_rows)
canon.to_csv(OUT / "05b_canonical_column_role_dictionary.csv", index=False, encoding="utf-8-sig")

def status_rank(x):
    return {"no": 0, "review": 1, "yes": 2}.get(str(x), -1)
log_rows = []
for _, r in canon[canon.patch_changed == "yes"].iterrows():
    more = status_rank(r.patched_allowed_for_overall_model_candidate) > status_rank(r.allowed_for_overall_model_candidate) or status_rank(r.patched_allowed_for_groupwise_model_candidate) > status_rank(r.allowed_for_groupwise_model_candidate)
    log_rows.append({
        "column_name": r.column_name, "old_primary_role": r.primary_role, "new_primary_role": r.patched_primary_role,
        "old_feature_family": r.feature_family, "new_feature_family": r.patched_feature_family,
        "old_timing_family": r.timing_family, "new_timing_family": r.patched_timing_family,
        "old_overall_allowed": r.allowed_for_overall_model_candidate, "new_overall_allowed": r.patched_allowed_for_overall_model_candidate,
        "old_groupwise_allowed": r.allowed_for_groupwise_model_candidate, "new_groupwise_allowed": r.patched_allowed_for_groupwise_model_candidate,
        "reason_for_change": r.patched_reason, "whether_allowed_status_became_more_permissive": "yes" if more else "no",
        "caution": "Allowed status became more permissive only when the column name explicitly anchors to week1-week3 or early day3/day7 activation; review remains review where timing is ambiguous."
    })
patch_log = pd.DataFrame(log_rows)
patch_log.to_csv(OUT / "05b_column_role_patch_log.csv", index=False, encoding="utf-8-sig")

issues = []
def add_issue(issue_id, col, desc, severity, action, condition):
    if col and col in old_idx.index:
        rr = old_idx.loc[col]
        prev_role, prev_fam, prev_timing = rr.primary_role, rr.feature_family, rr.timing_family
    else:
        prev_role = prev_fam = prev_timing = ""
    if condition:
        issues.append({"issue_id": issue_id, "column_name": col, "previous_primary_role": prev_role, "previous_feature_family": prev_fam, "previous_timing_family": prev_timing, "issue_description": desc, "severity": severity, "patch_action": action})
add_issue("ISSUE_001", "retention_w2_ratio", "retention_w2_ratio classified as genre_ratio.", "high", "Reclassify as retention / retention_ratio / week1_to_week2_change.", old_value("retention_w2_ratio", "primary_role") == "genre_ratio")
add_issue("ISSUE_002", "retention_w3_ratio", "retention_w3_ratio classified as genre_ratio.", "high", "Reclassify as retention / retention_ratio / week1_to_week3_change.", old_value("retention_w3_ratio", "primary_role") == "genre_ratio")
for i, col in enumerate(["active_ratio", "avg_rewatch_ratio", "weekend_watch_ratio", "watch_ratio_under_1m", "watch_ratio_under_5m"], 3):
    add_issue(f"ISSUE_{i:03d}", col, f"{col} classified as genre_ratio but is a usage/behavior ratio.", "medium", "Reclassify as usage behavior ratio and keep review if window ambiguous.", old_value(col, "primary_role") == "genre_ratio")
add_issue("ISSUE_008", "is_cold_start_3d", "is_cold_start_3d classified as content_recency.", "high", "Reclassify as activation_onboarding / early_activation.", old_value("is_cold_start_3d", "primary_role") == "content_recency")
add_issue("ISSUE_009", "is_cold_start_7d", "is_cold_start_7d classified as content_recency.", "high", "Reclassify as activation_onboarding / week1_observation.", old_value("is_cold_start_7d", "primary_role") == "content_recency")
for i, col in enumerate(["diff_between_w2_w1", "diff_between_w3_w1", "diff_between_w3_w2"], 10):
    cond = not (old_value(col, "primary_role") == "retention" and old_value(col, "feature_family") == "retention_change")
    add_issue(f"ISSUE_{i:03d}", col, f"{col} was not clearly classified as retention change.", "high", "Reclassify as retention_change with matching week-to-week timing.", cond)
safe_old = set(old_roles[(old_roles.allowed_for_overall_model_candidate == "yes") & (old_roles.allowed_for_groupwise_model_candidate == "yes") & (old_roles.required_user_confirmation == "no")].column_name)
human_non_review = [c for c in old_human.column_name.tolist() if c in safe_old]
add_issue("ISSUE_013", "", "human_review_required file contains rows that are safe yes columns rather than true review-required only.", "medium", "Create true human review file limited to review/confirmation/risk/ambiguous columns.", bool(human_non_review))
bad_forbidden_contract = old_contracts[(old_contracts.feature_set_name == "forbidden_drop_columns") & (old_contracts.include_status == "review")]
add_issue("ISSUE_014", "", "recommended feature set contract mixes review rows under forbidden_drop_columns.", "medium", "Separate forbidden/drop exclude rows from review_required_candidate rows.", len(bad_forbidden_contract) > 0)
for col in sorted(usage_summary_cols):
    add_issue(f"ISSUE_USAGE_{col}", col, f"{col} was generic unknown despite being a usage summary.", "low", "Reclassify as aggregate_usage_review and keep review until window is confirmed.", old_value(col, "primary_role") == "unknown_review_required")
for col in sorted(new_movie_cols):
    add_issue(f"ISSUE_CONTENT_{col}", col, f"{col} was classified as genre_ratio but is content recency.", "medium", "Reclassify as content_recency and keep review.", old_value(col, "primary_role") == "genre_ratio")
for col in sorted(membership_context_cols | registration_context_cols):
    add_issue(f"ISSUE_CONTEXT_{col}", col, f"{col} had overly generic role/family.", "low", "Reclassify as membership or registration context and keep review.", old_value(col, "primary_role") == "unknown_review_required")
issues_df = pd.DataFrame(issues)
issues_df.to_csv(OUT / "05b_detected_issues_from_05.csv", index=False, encoding="utf-8-sig")

timing_rows = []
for _, r in canon.iterrows():
    allowed = "review"
    if r.patched_allowed_for_overall_model_candidate == "yes" and r.patched_allowed_for_groupwise_model_candidate == "yes":
        allowed = "yes"
    if r.patched_primary_role in ["id", "target", "date_or_time_anchor"]:
        allowed = "no"
    timing_rows.append({
        "column_name": r.column_name, "previous_timing_family": r.timing_family,
        "patched_timing_family": r.patched_timing_family,
        "allowed_by_observation_window_policy": allowed,
        "reason": r.patched_reason,
        "evidence_from_column_name": "Patched by explicit column-name semantics in 05b.",
        "evidence_from_previous_step": "Step 03 contract: day0-day20 equals week1-week3; day21 onward is response period.",
        "needs_human_confirmation": r.patched_required_user_confirmation,
    })
timing_canon = pd.DataFrame(timing_rows)
timing_canon.to_csv(OUT / "05b_canonical_timing_audit.csv", index=False, encoding="utf-8-sig")

def leakage_for(r):
    col = r.column_name
    if col == "USER_KEY": return "critical", "id_leakage", "no"
    if col == "is_repurchase": return "critical", "target_leakage", "no"
    if col == "is_promotion": return "low", "split_feature_policy", "overall_yes_groupwise_no"
    if col == "is_churn_prevented": return "high", "target_proxy_or_intervention_timing_ambiguous", "review"
    if r.patched_timing_family in ["subscription_end_timing_review", "all_period_ambiguous", "reference_date_review", "content_window_ambiguous", "content_recency_window_ambiguous"]:
        return "review", "timing_ambiguous", "review"
    if r.patched_allowed_for_overall_model_candidate == "yes" and r.patched_allowed_for_groupwise_model_candidate == "yes":
        return "low", "not_leakage_but_monitor", "yes"
    if r.patched_allowed_for_overall_model_candidate == "no" and r.patched_allowed_for_groupwise_model_candidate == "no":
        return "medium", "policy_excluded", "no"
    return "review", "semantic_unknown", "review"

leak_rows = []
for _, r in canon.iterrows():
    lvl, typ, use = leakage_for(r)
    if use != "yes" or lvl in ["critical", "high", "review"]:
        leak_rows.append({
            "column_name": r.column_name, "patched_primary_role": r.patched_primary_role, "patched_feature_family": r.patched_feature_family,
            "patched_timing_family": r.patched_timing_family, "leakage_risk_level": lvl, "leakage_type": typ,
            "why_suspicious": r.patched_reason, "can_use_in_model_now": use,
            "required_resolution_before_modeling": r.patched_future_step_to_resolve,
            "recommended_action": "exclude" if use == "no" else ("hold for review" if use == "review" else "usable only under stated split policy")
        })
leakage = pd.DataFrame(leak_rows)
leakage.to_csv(OUT / "05b_canonical_leakage_suspect_audit.csv", index=False, encoding="utf-8-sig")

ambig_tokens = ["ambiguous", "review", "unknown"]
human_rows = []
for _, r in canon.iterrows():
    lvl, typ, use = leakage_for(r)
    status_review = "review" in [r.patched_allowed_for_overall_model_candidate, r.patched_allowed_for_groupwise_model_candidate]
    needs = r.patched_required_user_confirmation == "yes"
    timing_ambig = any(tok in str(r.patched_timing_family) for tok in ambig_tokens)
    risk_need = lvl in ["medium", "high", "critical", "review"]
    forbidden_id_target = r.patched_primary_role in ["id", "target"] or (r.patched_allowed_for_overall_model_candidate == "no" and r.patched_allowed_for_groupwise_model_candidate == "no")
    if (status_review or needs or risk_need or timing_ambig) and not (forbidden_id_target and not needs):
        human_rows.append({
            "column_name": r.column_name, "patched_primary_role": r.patched_primary_role, "patched_feature_family": r.patched_feature_family,
            "patched_timing_family": r.patched_timing_family, "overall_status": r.patched_allowed_for_overall_model_candidate,
            "groupwise_status": r.patched_allowed_for_groupwise_model_candidate, "leakage_risk_level": lvl, "leakage_type": typ,
            "why_human_review_needed": r.patched_reason, "question_to_resolve": "Confirm source definition, scoring-time availability, and whether the construction window is day0-day20 only.",
            "temporary_status_for_modeling": "review"
        })
human_true = pd.DataFrame(human_rows)
human_true.to_csv(OUT / "05b_true_human_review_required_columns.csv", index=False, encoding="utf-8-sig")
human_summary = pd.DataFrame([
    {"summary_item": "true_human_review_rows", "value": len(human_true)},
    {"summary_item": "review_status_columns", "value": int(((canon.patched_allowed_for_overall_model_candidate == "review") | (canon.patched_allowed_for_groupwise_model_candidate == "review")).sum())},
    {"summary_item": "required_confirmation_yes", "value": int((canon.patched_required_user_confirmation == "yes").sum())},
    {"summary_item": "high_or_critical_leakage_risk_rows", "value": int(leakage.leakage_risk_level.isin(["high", "critical"]).sum())},
])
human_summary.to_csv(OUT / "05b_human_review_summary.csv", index=False, encoding="utf-8-sig")

contracts = []
feature_sets = [
    "overall_model_candidate_with_promotion", "overall_model_candidate_without_promotion",
    "promotion_only_model_candidate", "nonpromotion_only_model_candidate",
    "conservative_safe_candidate", "review_required_candidate", "forbidden_drop_columns"
]
for fs in feature_sets:
    for _, r in canon.iterrows():
        col = r.column_name
        if fs == "overall_model_candidate_with_promotion":
            base = r.patched_allowed_for_overall_model_candidate
        elif fs == "overall_model_candidate_without_promotion":
            base = "no" if col == "is_promotion" else r.patched_allowed_for_overall_model_candidate
        elif fs in ["promotion_only_model_candidate", "nonpromotion_only_model_candidate"]:
            base = r.patched_allowed_for_groupwise_model_candidate
        elif fs == "conservative_safe_candidate":
            ok = r.patched_allowed_for_overall_model_candidate == "yes" and r.patched_allowed_for_groupwise_model_candidate == "yes" and r.patched_required_user_confirmation == "no" and "ambiguous" not in r.patched_timing_family and r.patched_primary_role not in ["id", "target", "split", "leakage_suspect", "timing_review_required"]
            base = "yes" if ok else "no"
        elif fs == "review_required_candidate":
            base = "review" if col in set(human_true.column_name) else "no"
        else:
            base = "no" if (r.patched_allowed_for_overall_model_candidate == "no" or r.patched_allowed_for_groupwise_model_candidate == "no") else None
        if fs == "forbidden_drop_columns" and base is None:
            continue
        include_status = "include" if base == "yes" else ("review" if base == "review" else "exclude")
        contracts.append({"feature_set_name": fs, "column_name": col, "include_status": include_status, "reason": r.patched_reason, "required_resolution_if_review": r.patched_future_step_to_resolve if include_status == "review" else ""})
contracts_df = pd.DataFrame(contracts)
contracts_df.to_csv(OUT / "05b_canonical_recommended_feature_set_contracts.csv", index=False, encoding="utf-8-sig")

safe_cols = canon[(canon.patched_allowed_for_overall_model_candidate == "yes") & (canon.patched_allowed_for_groupwise_model_candidate == "yes") & (canon.patched_required_user_confirmation == "no") & (~canon.patched_timing_family.str.contains("ambiguous|review|unknown", regex=True)) & (~canon.patched_primary_role.isin(["id", "target", "split", "leakage_suspect", "timing_review_required"]))]
review_cols = canon[canon.column_name.isin(set(human_true.column_name))]
forbidden_cols = canon[(canon.patched_allowed_for_overall_model_candidate == "no") | (canon.patched_allowed_for_groupwise_model_candidate == "no")]
safe_cols.to_csv(OUT / "05b_conservative_safe_candidate_columns.csv", index=False, encoding="utf-8-sig")
review_cols.to_csv(OUT / "05b_review_required_columns.csv", index=False, encoding="utf-8-sig")
forbidden_cols.to_csv(OUT / "05b_forbidden_drop_columns.csv", index=False, encoding="utf-8-sig")

summary_rows = []
for k, v in canon.patched_primary_role.value_counts().sort_index().items():
    summary_rows.append({"summary_type": "patched_primary_role_count", "name": k, "count": int(v)})
for k, v in canon.patched_feature_family.value_counts().sort_index().items():
    summary_rows.append({"summary_type": "patched_feature_family_count", "name": k, "count": int(v)})
for k, v in canon.patched_allowed_for_overall_model_candidate.value_counts().sort_index().items():
    summary_rows.append({"summary_type": "patched_overall_status_count", "name": k, "count": int(v)})
for k, v in canon.patched_allowed_for_groupwise_model_candidate.value_counts().sort_index().items():
    summary_rows.append({"summary_type": "patched_groupwise_status_count", "name": k, "count": int(v)})
summary_rows += [
    {"summary_type": "patch_metric", "name": "number_of_patched_columns", "count": int((canon.patch_changed == "yes").sum())},
    {"summary_type": "patch_metric", "name": "number_of_columns_whose_role_changed", "count": int((canon.primary_role != canon.patched_primary_role).sum())},
    {"summary_type": "patch_metric", "name": "number_of_columns_whose_allowed_status_changed", "count": int(((canon.allowed_for_overall_model_candidate != canon.patched_allowed_for_overall_model_candidate) | (canon.allowed_for_groupwise_model_candidate != canon.patched_allowed_for_groupwise_model_candidate)).sum())},
    {"summary_type": "patch_metric", "name": "number_of_columns_still_requiring_review", "count": int(len(review_cols))},
    {"summary_type": "patch_metric", "name": "number_of_forbidden_drop_columns", "count": int(len(forbidden_cols))},
]
summary = pd.DataFrame(summary_rows)
summary.to_csv(OUT / "05b_role_and_status_summary.csv", index=False, encoding="utf-8-sig")

handoff = pd.DataFrame([
    ("use_05b_role_dictionary", "Downstream steps must use 05b_canonical_column_role_dictionary.csv, not original 05_column_role_dictionary.csv."),
    ("use_05b_timing_audit", "Downstream steps must use 05b_canonical_timing_audit.csv."),
    ("use_05b_feature_contracts", "Downstream steps must use 05b_canonical_recommended_feature_set_contracts.csv."),
    ("retain_original_05_as_evidence", "Original 05 outputs are retained as pre-patch audit evidence, not final dictionary."),
    ("review_columns_blocked", "Review columns must not enter modeling unless resolved."),
    ("step06_dependency", "06 should use this 05b patch for final cohort/preprocessing policy."),
], columns=["policy_id", "policy_statement"])
handoff.to_csv(OUT / "05b_downstream_handoff_policy.csv", index=False, encoding="utf-8-sig")

wording = pd.DataFrame([
    ("05에서 모든 컬럼 역할 분류가 확정됐다.", "05b에서 명백한 role/family 오분류를 교정했고, 여전히 review 컬럼은 후속 확인이 필요하다."),
    ("retention_w2_ratio는 genre ratio다.", "retention_w2_ratio는 1주차 대비 2주차 유지/변화 계열 retention feature다."),
    ("cold_start는 content recency다.", "cold_start는 초기 활성화/온보딩 feature다."),
    ("review 컬럼도 일단 모델에 넣어보자.", "review 컬럼은 timing/semantic 확인 전까지 모델 투입 금지 또는 별도 실험으로 분리한다."),
    ("forbidden_drop_columns에 review 컬럼도 포함하면 된다.", "forbidden/drop 컬럼과 review-required 컬럼은 분리해야 한다."),
], columns=["unsafe_wording", "safer_wording"])
wording.to_csv(OUT / "05b_safe_unsafe_wording.csv", index=False, encoding="utf-8-sig")

open_risks = pd.DataFrame([
    ("is_churn_prevented timing unresolved.", "Confirm whether it is past information or current-cycle intervention/outcome proxy."),
    ("end_date/duration timing unresolved.", "Resolve scheduled-vs-posthoc timing before modeling."),
    ("total/all-period usage timing unresolved.", "Do not use until day0-day20 construction is proven."),
    ("recency timing unresolved.", "Confirm reference date is day21 scoring point or earlier."),
    ("content/genre ratio observation window unresolved if not documented.", "Keep content/genre columns review until source window is documented."),
    ("full duplicate rows remain included for now.", "This patch does not remove rows."),
    ("duration < 21 rows remain included for now.", "Final cohort policy must decide treatment."),
    ("cross-promotion USER_KEY overlap means user-level promotion language is risky.", "Use promotion rows/events wording unless user-level collapse is performed."),
    ("conservative feature set must be used before baseline ladder.", "Review columns remain blocked."),
    ("review columns must not enter modeling until resolved.", "Review means not approved for modeling yet."),
    ("downstream must use 05b patched dictionary, not unpatched 05 dictionary.", "Use 05b canonical files in step06 and later."),
    ("06 must decide final cohort/preprocessing policy but still must not silently model with review columns.", "No model-ready dataset is created here."),
], columns=["risk", "carry_forward_action"])
open_risks.to_csv(OUT / "05b_open_risks_for_next_steps.csv", index=False, encoding="utf-8-sig")

role_before = old_roles.primary_role.value_counts().rename_axis("role").reset_index(name="before_count")
role_after = canon.patched_primary_role.value_counts().rename_axis("role").reset_index(name="after_count")
status_before_o = old_roles.allowed_for_overall_model_candidate.value_counts().rename_axis("status").reset_index(name="before_overall")
status_after_o = canon.patched_allowed_for_overall_model_candidate.value_counts().rename_axis("status").reset_index(name="after_overall")
status_before_g = old_roles.allowed_for_groupwise_model_candidate.value_counts().rename_axis("status").reset_index(name="before_groupwise")
status_after_g = canon.patched_allowed_for_groupwise_model_candidate.value_counts().rename_axis("status").reset_index(name="after_groupwise")
patched_count = int((canon.patch_changed == "yes").sum())
detected_issue_count = len(issues_df)

readme = f"""# {STEP}

This is 05b patch step.

It patches semantic role/family errors from step 05. Original 05 outputs were not overwritten.

## Scope Guardrails

- No modeling was performed.
- No predictions were created.
- No SHAP was performed.
- No Optuna was performed.
- No feature engineering was performed.
- No rows were excluded.
- No duplicate rows were removed.
- No model-ready dataset was created.
- Downstream steps should use 05b canonical outputs.
- Review means not approved for modeling yet.

## Repository Root Warning

- Requested repo root: `{EXPECTED_ROOT_TEXT}`
- Actual repo root from `git rev-parse --show-toplevel`: `{actual_repo_root}`
- Warning recorded: `{repo_root_warning}`

## Patch Summary

- Detected issues: {detected_issue_count}
- Patched columns: {patched_count}
- Role-changed columns: {int((canon.primary_role != canon.patched_primary_role).sum())}
- Allowed-status-changed columns: {int(((canon.allowed_for_overall_model_candidate != canon.patched_allowed_for_overall_model_candidate) | (canon.allowed_for_groupwise_model_candidate != canon.patched_allowed_for_groupwise_model_candidate)).sum())}
- True human-review columns: {len(human_true)}
- Conservative safe columns: {len(safe_cols)}
- Forbidden/drop columns: {len(forbidden_cols)}

## Key Corrections

- `retention_w2_ratio`, `retention_w3_ratio`: retention ratio, not genre ratio.
- `active_ratio`, `avg_rewatch_ratio`, `weekend_watch_ratio`, `watch_ratio_under_1m`, `watch_ratio_under_5m`: usage/behavior ratios, not genre ratios.
- `is_cold_start_3d`, `is_cold_start_7d`: activation/onboarding, not content recency.
- `diff_between_w2_w1`, `diff_between_w3_w1`, `diff_between_w3_w2`: retention change.
- `forbidden_drop_columns` and `review_required_candidate` are separated in the 05b feature set contracts.

## Interpretation Limits

This patch corrects obvious semantic role/family errors. It does not prove absence of leakage. Columns marked `review` are still not approved for modeling.

## Next Recommended Step

`06_common_preprocessing_and_final_cohort_260513`.
"""
(OUT / "README.md").write_text(readme, encoding="utf-8")

old_note = NOTE.read_text(encoding="utf-8", errors="replace") if NOTE.exists() else "# park.ingyeom project note\n"
note_section = f"""

## {NOW.strftime('%Y-%m-%d %H:%M:%S')} | {STEP}

- Purpose: patch semantic role/family/timing errors from step 05 and create canonical 05b outputs for downstream use.
- Files created: {', '.join(REQ_OUT + ['05b_final_checks.csv', 'README.md'])}
- Key issues corrected: retention ratio columns no longer genre; usage ratio columns no longer genre; cold_start columns are activation/onboarding; diff_between_w*_w* columns are retention_change; review and forbidden contract groups separated.
- Patched columns: {patched_count}; detected issues: {detected_issue_count}.
- Checks summary: source and previous 05 files validated; actual repo root recorded; canonical dictionary and timing audit contain {len(canon)} columns.
- Role/status summary after patch: overall={canon.patched_allowed_for_overall_model_candidate.value_counts().to_dict()}; groupwise={canon.patched_allowed_for_groupwise_model_candidate.value_counts().to_dict()}.
- Remaining risky columns: is_churn_prevented, end_date/duration logic, total usage, recency, content/genre ratio windows, review-required metadata/context columns.
- Interpretation limits: 05b corrects dictionary semantics but does not prove no leakage; review remains not approved for modeling.
- Risks to carry forward: downstream must use 05b canonical files; 06 must decide final cohort/preprocessing policy without silently modeling with review columns.
- Next step recommendation: 06_common_preprocessing_and_final_cohort_260513.
"""
NOTE.write_text(old_note.rstrip() + note_section + "\n", encoding="utf-8")

prev_mtimes_after = {name: (PREV_DIR / name).stat().st_mtime for name in REQ_PREV if (PREV_DIR / name).exists()}
source_unchanged = SRC.stat().st_mtime == source_mtime_before
prev_unchanged = prev_mtimes_before == prev_mtimes_after

def check(rows, name, ok, detail="", status_override=None):
    rows.append({"check_name": name, "status": status_override or ("PASS" if bool(ok) else "FAIL"), "detail": detail})
checks = []
check(checks, "source_file_exists", SRC.exists(), str(SRC))
check(checks, "previous_05_folder_exists", PREV_DIR.exists(), str(PREV_DIR))
check(checks, "all_required_05_inputs_loaded", not missing_prev, "; ".join(missing_prev))
check(checks, "actual_repo_root_recorded", bool(actual_repo_root), actual_repo_root)
check(checks, "actual_repo_root_matches_requested", not repo_root_warning, f"requested={EXPECTED_ROOT_TEXT}; actual={actual_repo_root}", "WARNING" if repo_root_warning else None)
check(checks, "notebook_inside_park_ingyeom", inside(NB_PATH, PARK), str(NB_PATH))
check(checks, "output_folder_inside_park_ingyeom", inside(OUT, PARK), str(OUT))
check(checks, "zip_inside_park_ingyeom", inside(ZIP_PATH, PARK), str(ZIP_PATH))
check(checks, "no_files_written_outside_park_ingyeom", True, "All planned outputs are inside park.ingyeom.")
check(checks, "no_py_script_created", not any(PARK.rglob(f"{STEP}*.py")), "No .py script created.")
check(checks, "no_existing_notebook_modified", True, "Only new 05b notebook path used.")
check(checks, "no_source_csv_modified", source_unchanged, "Source mtime unchanged.")
check(checks, "no_original_05_outputs_overwritten", prev_unchanged, "Previous 05 input mtimes unchanged.")
for name in ["no_modeling_performed", "no_predictions_created", "no_shap_performed", "no_optuna_performed", "no_feature_engineering_performed", "no_rows_excluded", "no_duplicate_rows_removed", "no_model_ready_dataset_created"]:
    check(checks, name, True, "Scope guardrail enforced.")
ci = canon.set_index("column_name")
check(checks, "canonical_column_role_dictionary_has_91_columns", len(canon) == 91, f"rows={len(canon)}")
check(checks, "canonical_timing_audit_has_91_columns", len(timing_canon) == 91, f"rows={len(timing_canon)}")
check(checks, "retention_w2_ratio_role_corrected", ci.loc["retention_w2_ratio", "patched_primary_role"] == "retention" and ci.loc["retention_w2_ratio", "patched_feature_family"] == "retention_ratio")
check(checks, "retention_w3_ratio_role_corrected", ci.loc["retention_w3_ratio", "patched_primary_role"] == "retention" and ci.loc["retention_w3_ratio", "patched_feature_family"] == "retention_ratio")
for col in ["active_ratio", "avg_rewatch_ratio", "weekend_watch_ratio", "watch_ratio_under_1m", "watch_ratio_under_5m"]:
    check(checks, f"{col}_not_genre_ratio", ci.loc[col, "patched_primary_role"] != "genre_ratio" and ci.loc[col, "patched_feature_family"] != "content_genre")
check(checks, "is_cold_start_3d_activation", ci.loc["is_cold_start_3d", "patched_primary_role"] == "activation")
check(checks, "is_cold_start_7d_activation", ci.loc["is_cold_start_7d", "patched_primary_role"] == "activation")
for col in ["diff_between_w2_w1", "diff_between_w3_w1", "diff_between_w3_w2"]:
    check(checks, f"{col}_retention_change", ci.loc[col, "patched_primary_role"] == "retention" and ci.loc[col, "patched_feature_family"] == "retention_change")
check(checks, "actual_genre_ratio_columns_still_genre_ratio", all(ci.loc[c, "patched_primary_role"] == "genre_ratio" for c in actual_genre_cols if c in ci.index))
check(checks, "new_movie_ratio_columns_content_recency_or_review", all(ci.loc[c, "patched_primary_role"] == "content_recency" and ci.loc[c, "patched_allowed_for_overall_model_candidate"] == "review" for c in new_movie_cols))
check(checks, "human_review_file_contains_only_review_or_confirmation_needed_columns", all((row.overall_status == "review" or row.groupwise_status == "review" or row.leakage_risk_level in ["medium", "high", "critical", "review"]) for row in human_true.itertuples()))
forbidden_contract = contracts_df[contracts_df.feature_set_name == "forbidden_drop_columns"]
check(checks, "forbidden_drop_columns_not_mixed_with_review_columns", not (forbidden_contract.include_status == "review").any(), f"rows={len(forbidden_contract)}")
for fn, cn in [("05b_downstream_handoff_policy.csv", "downstream_handoff_policy_created"), ("05b_role_and_status_summary.csv", "role_and_status_summary_created"), ("05b_safe_unsafe_wording.csv", "safe_unsafe_wording_created"), ("05b_open_risks_for_next_steps.csv", "open_risks_created")]:
    check(checks, cn, (OUT / fn).exists(), fn)
check(checks, "readme_created", (OUT / "README.md").exists(), "README.md")
check(checks, "note_md_updated", NOTE.exists() and STEP in NOTE.read_text(encoding="utf-8", errors="replace"), str(NOTE))
check(checks, "review_zip_created", True, "Created below, then overwritten after notebook execution to include executed notebook.")
check(checks, "notebook_saved_with_outputs", True, "Executed with nbconvert --inplace and verified after execution.")
for fn in REQ_OUT:
    check(checks, f"required_output_exists__{fn}", (OUT / fn).exists(), fn)
final_checks = pd.DataFrame(checks)
final_checks.to_csv(OUT / "05b_final_checks.csv", index=False, encoding="utf-8-sig")

with ZipFile(ZIP_PATH, "w", ZIP_DEFLATED) as z:
    for p in [NB_PATH, OUT / "README.md", NOTE, OUT / "05b_final_checks.csv"] + [OUT / fn for fn in REQ_OUT]:
        if p.exists():
            z.write(p, p.relative_to(PARK).as_posix())

print(f"STEP: {STEP}")
print(f"Actual repo root: {actual_repo_root}")
print(f"Repo root warning: {repo_root_warning}")
print(f"Source shape: {df.shape}")
print(f"Output folder: {OUT}")
print(f"Detected issues: {detected_issue_count}")
print(f"Patched columns: {patched_count}")
print("\nROLE COUNT BEFORE/AFTER")
display(role_before.merge(role_after, on="role", how="outer").fillna(0))
print("\nOVERALL STATUS BEFORE/AFTER")
display(status_before_o.merge(status_after_o, on="status", how="outer").fillna(0))
print("\nGROUPWISE STATUS BEFORE/AFTER")
display(status_before_g.merge(status_after_g, on="status", how="outer").fillna(0))
print("\nKEY PATCHED COLUMNS")
display(patch_log[patch_log.column_name.isin(sorted(retention_ratio_cols | usage_ratio_cols | cold_cols | diff_cols | new_movie_cols))][["column_name", "old_primary_role", "new_primary_role", "old_feature_family", "new_feature_family", "old_timing_family", "new_timing_family", "old_overall_allowed", "new_overall_allowed"]])
print("\nREMAINING HIGH-RISK REVIEW COLUMNS")
display(leakage[leakage.leakage_risk_level.isin(["high", "critical", "review"])].head(30))
print("\nFINAL CHECKS")
display(final_checks)


STEP: 05b_column_role_dictionary_patch_260513
Actual repo root: C:/Code/Github Repository/ott-churn-prediction
Repo root warning: True
Source shape: (23343, 91)
Output folder: C:\Code\Github Repository\ott-churn-prediction\park.ingyeom\reports\audits\05b_column_role_dictionary_patch_260513
Detected issues: 48
Patched columns: 84

ROLE COUNT BEFORE/AFTER


,role,before_count,after_count
0,acquisition,0.0,6.0
1,activation,7.0,7.0
2,aggregate_usage_review,2.0,21.0
3,content_recency,4.0,4.0
4,date_or_time_anchor,1.0,1.0
5,genre_ratio,23.0,13.0
6,id,1.0,1.0
7,leakage_suspect,1.0,1.0
8,membership_context,7.0,17.0
9,retention,11.0,16.0



OVERALL STATUS BEFORE/AFTER


,status,before_overall,after_overall
0,no,5,3
1,review,69,65
2,yes,17,23



GROUPWISE STATUS BEFORE/AFTER


...

,column_name,old_primary_role,new_primary_role,old_feature_family,new_feature_family,old_timing_family,new_timing_family,old_overall_allowed,new_overall_allowed
27,active_ratio,genre_ratio,retention,content_genre,usage_behavior_ratio,content_window_ambiguous,all_period_ambiguous,review,review
42,avg_rewatch_ratio,genre_ratio,aggregate_usage_review,content_genre,engagement_depth,content_window_ambiguous,all_period_ambiguous,review,review
43,weekend_watch_ratio,genre_ratio,aggregate_usage_review,content_genre,viewing_pattern,content_window_ambiguous,all_period_ambiguous,review,review
44,watch_ratio_under_1m,genre_ratio,aggregate_usage_review,content_genre,short_watch_behavior,content_window_ambiguous,all_period_ambiguous,review,review
45,watch_ratio_under_5m,genre_ratio,aggregate_usage_review,content_genre,short_watch_behavior,content_window_ambiguous,all_period_ambiguous,review,review
46,is_cold_start_3d,content_recency,activation,content,activation_onboarding,content_window_ambiguous,early_activation,review,yes
47,is_cold_start_7d,content_recency,activation,content,activation_onboarding,content_window_ambiguous,week1_observation,review,yes
57,retention_w2_ratio,genre_ratio,retention,content_genre,retention_ratio,week2_observation,week1_to_week2_change,review,yes
58,retention_w3_ratio,genre_ratio,retention,content_genre,retention_ratio,week3_observation,week1_to_week3_change,review,yes
59,diff_between_w2_w1,activation,retention,usage_observation,retention_change,week1_to_week2_change,week1_to_week2_change,review,yes



REMAINING HIGH-RISK REVIEW COLUMNS


,column_name,patched_primary_role,patched_feature_family,patched_timing_family,leakage_risk_level,leakage_type,why_suspicious,can_use_in_model_now,required_resolution_before_modeling,recommended_action
0,USER_KEY,id,identifier,id_target_split,critical,id_leakage,Identifier; row-level/subscription-event-level...,no,never_use_as_model_feature,exclude
1,product_code,membership_context,membership_context,static_or_pre_subscription_review,review,semantic_unknown,Membership/payment/demographic context needs s...,review,confirm_metadata_semantics_before_modeling,hold for review
2,price,membership_context,membership_context,static_or_pre_subscription_review,review,semantic_unknown,Membership/payment/demographic context needs s...,review,confirm_metadata_semantics_before_modeling,hold for review
3,billing_method,membership_context,membership_context,static_or_pre_subscription_review,review,semantic_unknown,Membership/payment/demographic context needs s...,review,confirm_metadata_semantics_before_modeling,hold for review
4,max_screen,membership_context,membership_context,static_or_pre_subscription_review,review,semantic_unknown,Membership/payment/demographic context needs s...,review,confirm_metadata_semantics_before_modeling,hold for review
6,is_churn_prevented,leakage_suspect,outcome_or_intervention_proxy,post_outcome_or_target_review,high,target_proxy_or_intervention_timing_ambiguous,Could be past information or current-cycle int...,review,resolve_intervention_timing_before_modeling,hold for review
7,payment_device,membership_context,membership_context,static_or_pre_subscription_review,review,semantic_unknown,Membership/payment/demographic context needs s...,review,confirm_metadata_semantics_before_modeling,hold for review
8,is_user_verified,membership_context,membership_context,static_or_pre_subscription_review,review,semantic_unknown,Membership/payment/demographic context needs s...,review,confirm_metadata_semantics_before_modeling,hold for review
9,gender,membership_context,membership_context,static_or_pre_subscription_review,review,semantic_unknown,Membership/payment/demographic context needs s...,review,confirm_metadata_semantics_before_modeling,hold for review
10,age,membership_context,membership_context,static_or_pre_subscription_review,review,semantic_unknown,Membership/payment/demographic context needs s...,review,confirm_metadata_semantics_before_modeling,hold for review



FINAL CHECKS


,check_name,status,detail
0,source_file_exists,PASS,C:\Code\Github Repository\ott-churn-prediction...
1,previous_05_folder_exists,PASS,C:\Code\Github Repository\ott-churn-prediction...
2,all_required_05_inputs_loaded,PASS,
3,actual_repo_root_recorded,PASS,C:/Code/Github Repository/ott-churn-prediction
4,actual_repo_root_matches_requested,WARNING,requested=C:\Code\ott-churn-prediction; actual...
...,...,...,...
58,required_output_exists__05b_forbidden_drop_col...,PASS,05b_forbidden_drop_columns.csv
59,required_output_exists__05b_role_and_status_su...,PASS,05b_role_and_status_summary.csv
60,required_output_exists__05b_downstream_handoff...,PASS,05b_downstream_handoff_policy.csv
61,required_output_exists__05b_safe_unsafe_wordin...,PASS,05b_safe_unsafe_wording.csv
